# Entrenamiento U-Net

Segmentación semántica de chips Sentinel-2 para los niveles 1 y 2 de Corine Land Cover. El flujo valida la correspondencia espacial entre cada imagen y su máscara, conserva las bandas B2, B3, B4 y B8 y remapea las categorías temáticas a índices internos contiguos, excluyendo NoData de la salida del modelo.

> **Alcance metodológico:** U-Net realiza segmentación por píxel, mientras que ResNet-50 y DenseNet-121 realizan clasificación por chip; por ello, las métricas internas de entrenamiento no son directamente comparables y la comparación entre arquitecturas debe efectuarse sobre productos cartográficos generados con una referencia, extensión y unidad de evaluación comunes.

## Dependencias

In [ ]:
from __future__ import annotations

import gc
import json
import os
import random
import re
import warnings
from contextlib import nullcontext
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from tqdm.auto import tqdm

## Configuración

In [ ]:
DATASET_ROOT = Path(r"D:\Dataset")
METADATA_DIR = DATASET_ROOT / "metadata"
RUNS_DIR = Path(r"D:\Modelos_UNet")

LEVELS_TO_TRAIN = ["nivel_1", "nivel_2"]
CREATE_NEW_RUNS = True
RESUME_BY_LEVEL = {"nivel_1": None, "nivel_2": None}

REQUIRE_CUDA = True
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42
DETERMINISTIC = True

EPOCHS = 120
BATCH_SIZE = 1
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-4
OPTIMIZER_TYPE = "adamw"
NUM_WORKERS = 0
EARLY_STOPPING_PATIENCE = 20
USE_AMP = True
GRADIENT_ACCUMULATION_STEPS = 1

BASE_CHANNELS = 64
BILINEAR = False
DROPOUT = 0.0
BOTTLENECK_DROPOUT = 0.5

BAND_INDEXES = (1, 2, 3, 4)
EXPECTED_CHIP_BANDS = 4
REFLECTANCE_SCALE = 10000.0
REFLECTANCE_ADD_OFFSET = 0.0
USE_RASTER_SCALE_OFFSET = True
AUTO_SCALE_REFLECTANCE = True
STATS_SAMPLE_SIZE = 300

MASK_NODATA_CODE = 0
IGNORE_INDEX = -100
LOSS_TYPE = "ce_dice"
DICE_WEIGHT = 0.7
USE_CLASS_WEIGHTS = True
USE_WEIGHTED_SAMPLER = False
USE_AUGMENTATION = True

SAVE_EVERY_EPOCH = False
SAVE_TEST_PREDICTIONS = True
MAX_TEST_PREDICTIONS_TO_SAVE = 20
SAVE_REPORT_PLOTS = True

## Clases temáticas

In [ ]:
CLASES_NIVEL_1 = {
    0: "NoData",
    1: "2. Territorios agrícolas",
    2: "3. Bosques y áreas seminaturales",
    3: "5. Superficies de agua",
    4: "1. Territorios artificializados",
    5: "4. Áreas húmedas",
}

CLASES_NIVEL_2 = {
    0: "NoData",
    1: "2.4 Áreas agrícolas heterogéneas",
    2: "3.2 Áreas con vegetación herbácea o arbustiva",
    3: "3.1 Bosques",
    4: "2.3 Pastos",
    5: "5.1 Aguas continentales",
    6: "1.1 Zonas urbanizadas",
    7: "1.3 Zonas de extracción minera y escombreras",
    8: "4.1 Áreas húmedas continentales",
    9: "1.2 Zonas industriales o comerciales y redes de comunicación",
    10: "1.4 Zonas verdes artificializadas no agrícolas",
    11: "2.1 Cultivos transitorios",
    12: "2.2 Cultivos permanentes",
    13: "3.3 Áreas abiertas sin o con poca vegetación",
}

LEVEL_FOLDER_BY_NAME = {
    "nivel_1": "Nivel_1",
    "nivel_2": "Nivel_2",
    "metadata": "metadata",
}

def get_class_dict(level_name: str) -> dict[int, str]:
    if level_name == "nivel_1":
        return CLASES_NIVEL_1
    if level_name == "nivel_2":
        return CLASES_NIVEL_2
    raise ValueError(f"Nivel no reconocido: {level_name}")

## Validación de rutas, particiones y correspondencia espacial

In [ ]:
def set_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    if torch.cuda.is_available():
        torch.backends.cudnn.deterministic = DETERMINISTIC
        torch.backends.cudnn.benchmark = not DETERMINISTIC


def ensure_dir(path: str | Path) -> Path:
    path = Path(path)
    path.mkdir(parents=True, exist_ok=True)
    return path


def json_default(value):
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    raise TypeError(f"Objeto no serializable: {type(value).__name__}")


def save_json(data, path: str | Path) -> None:
    path = Path(path)
    ensure_dir(path.parent)
    with path.open("w", encoding="utf-8") as file:
        json.dump(data, file, indent=2, ensure_ascii=False, default=json_default)


def safe_name(text: str) -> str:
    normalized = (
        str(text)
        .translate(str.maketrans("áéíóúÁÉÍÓÚñÑ", "aeiouAEIOUnN"))
    )
    normalized = re.sub(r"[^A-Za-z0-9]+", "_", normalized)
    return re.sub(r"_+", "_", normalized).strip("_").lower()


def resolve_case_insensitive(path: Path) -> Path:
    if path.exists():
        return path

    parts = path.parts
    if not parts:
        return path

    current = Path(parts[0])
    for part in parts[1:]:
        if current.exists() and current.is_dir():
            try:
                match = next(
                    (item.name for item in current.iterdir() if item.name.lower() == part.lower()),
                    part,
                )
            except OSError:
                match = part
            current = current / match
        else:
            current = current / part
    return current


def resolve_dataset_path(value) -> str:
    if pd.isna(value):
        return ""

    raw = os.path.expandvars(os.path.expanduser(str(value).strip().strip('"').strip("'")))
    if not raw:
        return ""

    direct = resolve_case_insensitive(Path(raw))
    if direct.exists():
        return str(direct)

    parts = [part for part in raw.replace("\\", "/").split("/") if part not in {"", "."}]
    lower_parts = [part.lower() for part in parts]

    if "dataset" in lower_parts:
        index = len(lower_parts) - 1 - lower_parts[::-1].index("dataset")
        candidate = DATASET_ROOT.joinpath(*parts[index + 1 :])
    else:
        candidate = DATASET_ROOT.joinpath(*parts)

    relative_parts = list(candidate.relative_to(DATASET_ROOT).parts)
    if relative_parts:
        relative_parts[0] = LEVEL_FOLDER_BY_NAME.get(relative_parts[0].lower(), relative_parts[0])
    return str(resolve_case_insensitive(DATASET_ROOT.joinpath(*relative_parts)))


def get_splits_dir(level_name: str) -> Path:
    candidates = [
        METADATA_DIR / "splits" / level_name,
        METADATA_DIR / "splits" / LEVEL_FOLDER_BY_NAME.get(level_name, level_name),
    ]
    return next((path for path in candidates if path.exists()), candidates[0])


def find_split_file(splits_dir: Path, *names: str) -> Path:
    if not splits_dir.exists():
        raise FileNotFoundError(f"No existe la carpeta de splits: {splits_dir}")

    available = {item.name.lower(): item for item in splits_dir.iterdir() if item.is_file()}
    for name in names:
        if name.lower() in available:
            return available[name.lower()]

    raise FileNotFoundError(f"No se encontró ninguno de {names} en {splits_dir}")


def read_split_csv(csv_path: Path, split_name: str) -> pd.DataFrame:
    if not csv_path.exists() or csv_path.stat().st_size == 0:
        raise FileNotFoundError(f"CSV inexistente o vacío: {csv_path}")

    df = pd.read_csv(csv_path)
    columns = {str(column).strip().lower(): column for column in df.columns}

    image_column = next(
        (columns[name] for name in ("image_path", "image", "img_path", "img") if name in columns),
        None,
    )
    mask_column = next(
        (columns[name] for name in ("mask_path", "label_path", "mask", "label") if name in columns),
        None,
    )

    if image_column is None or mask_column is None:
        raise ValueError(
            f"El split {csv_path} debe contener image_path y mask_path; "
            f"columnas disponibles: {list(df.columns)}"
        )

    df = df.rename(columns={image_column: "image_path", mask_column: "mask_path"})
    df = df.dropna(subset=["image_path", "mask_path"]).copy()
    df["image_path"] = df["image_path"].apply(resolve_dataset_path)
    df["mask_path"] = df["mask_path"].apply(resolve_dataset_path)

    duplicate_count = int(df.duplicated(subset=["image_path", "mask_path"]).sum())
    if duplicate_count:
        warnings.warn(
            f"{split_name}: se eliminaron {duplicate_count} pares duplicados.",
            RuntimeWarning,
        )
        df = df.drop_duplicates(subset=["image_path", "mask_path"], keep="first")

    missing_images = [path for path in df["image_path"] if not Path(path).exists()]
    missing_masks = [path for path in df["mask_path"] if not Path(path).exists()]
    if missing_images or missing_masks:
        raise FileNotFoundError(
            f"{split_name}: faltan {len(missing_images)} imágenes y {len(missing_masks)} máscaras."
        )

    if df.empty:
        raise ValueError(f"El split {split_name} quedó vacío después de la validación.")

    return df.reset_index(drop=True)


def normalized_pair_set(df: pd.DataFrame) -> set[tuple[str, str]]:
    return {
        (
            str(Path(image)).replace("\\", "/").lower(),
            str(Path(mask)).replace("\\", "/").lower(),
        )
        for image, mask in zip(df["image_path"], df["mask_path"])
    }


def lineage_set(df: pd.DataFrame) -> set[str]:
    columns = {str(column).strip().lower(): column for column in df.columns}
    for candidate in ("source_chip_id", "original_chip_id", "chip_id"):
        if candidate in columns:
            values = df[columns[candidate]].dropna().astype(str).str.strip()
            return {value.lower() for value in values if value}
    return set()


def validate_split_disjointness(train: pd.DataFrame, val: pd.DataFrame, test: pd.DataFrame) -> None:
    pair_sets = {
        "train": normalized_pair_set(train),
        "val": normalized_pair_set(val),
        "test": normalized_pair_set(test),
    }
    pair_overlaps = {
        "train-val": pair_sets["train"] & pair_sets["val"],
        "train-test": pair_sets["train"] & pair_sets["test"],
        "val-test": pair_sets["val"] & pair_sets["test"],
    }

    lineage_sets = {
        "train": lineage_set(train),
        "val": lineage_set(val),
        "test": lineage_set(test),
    }
    lineage_overlaps = {
        "train-val": lineage_sets["train"] & lineage_sets["val"],
        "train-test": lineage_sets["train"] & lineage_sets["test"],
        "val-test": lineage_sets["val"] & lineage_sets["test"],
    }

    conflicts = {name: len(values) for name, values in pair_overlaps.items() if values}
    lineage_conflicts = {
        name: len(values)
        for name, values in lineage_overlaps.items()
        if values
    }

    if conflicts or lineage_conflicts:
        details = []
        if conflicts:
            details.append(
                "pares repetidos: "
                + ", ".join(f"{name}={count}" for name, count in conflicts.items())
            )
        if lineage_conflicts:
            details.append(
                "chips originales compartidos entre splits: "
                + ", ".join(f"{name}={count}" for name, count in lineage_conflicts.items())
            )
        raise ValueError(
            "Se detectó fuga de información entre entrenamiento, validación y prueba; "
            + "; ".join(details)
            + ". Los aumentos de un mismo chip deben permanecer en un único split."
        )


def validate_image_mask_pairs(
    dataframes: dict[str, pd.DataFrame],
    class_dict: dict[int, str],
) -> tuple[int, tuple[int, int]]:
    expected_shape = None
    observed_classes = set()

    for split_name, dataframe in dataframes.items():
        for image_path, mask_path in tqdm(
            zip(dataframe["image_path"], dataframe["mask_path"]),
            total=len(dataframe),
            desc=f"Validando {split_name}",
        ):
            with rasterio.open(image_path) as image_src, rasterio.open(mask_path) as mask_src:
                if image_src.count != EXPECTED_CHIP_BANDS:
                    raise ValueError(
                        f"{image_path} tiene {image_src.count} bandas; se esperaban {EXPECTED_CHIP_BANDS}."
                    )
                if max(BAND_INDEXES) > image_src.count:
                    raise ValueError(f"Índices de banda inválidos para {image_path}.")
                if (image_src.height, image_src.width) != (mask_src.height, mask_src.width):
                    raise ValueError(f"Dimensiones incompatibles: {image_path} y {mask_path}.")
                if image_src.crs != mask_src.crs:
                    raise ValueError(f"CRS incompatible: {image_path} y {mask_path}.")
                if not image_src.transform.almost_equals(mask_src.transform):
                    raise ValueError(f"Transformación espacial incompatible: {image_path} y {mask_path}.")

                shape = (image_src.height, image_src.width)
                if expected_shape is None:
                    expected_shape = shape
                elif shape != expected_shape:
                    raise ValueError(
                        f"Todos los chips de un nivel deben tener el mismo tamaño; "
                        f"se detectaron {expected_shape} y {shape}."
                    )

                mask = mask_src.read(1)
                observed_classes.update(int(value) for value in np.unique(mask))

    allowed = set(class_dict)
    unexpected = observed_classes - allowed
    if unexpected:
        raise ValueError(f"Las máscaras contienen códigos no definidos: {sorted(unexpected)}.")

    train_classes = set()
    for mask_path in dataframes["train"]["mask_path"]:
        with rasterio.open(mask_path) as src:
            train_classes.update(int(value) for value in np.unique(src.read(1)))
    missing_train = (allowed - {MASK_NODATA_CODE}) - train_classes
    if missing_train:
        raise ValueError(
            f"Clases sin píxeles en entrenamiento: {sorted(missing_train)}."
        )

    return EXPECTED_CHIP_BANDS, expected_shape


def read_splits(level_name: str, class_dict: dict[int, str]):
    splits_dir = get_splits_dir(level_name)
    train = read_split_csv(
        find_split_file(splits_dir, "train_samples.csv", "train.csv"),
        "train",
    )
    val = read_split_csv(
        find_split_file(splits_dir, "val_samples.csv", "val.csv"),
        "val",
    )
    test = read_split_csv(
        find_split_file(splits_dir, "test_samples.csv", "test.csv"),
        "test",
    )

    validate_split_disjointness(train, val, test)
    band_count, chip_shape = validate_image_mask_pairs(
        {"train": train, "val": val, "test": test},
        class_dict,
    )
    return train, val, test, splits_dir, band_count, chip_shape

## Dataset y estadísticas

Cada muestra conserva las bandas B2, B3, B4 y B8 y su máscara temática georreferenciada. La lectura aplica las escalas y los desplazamientos radiométricos del ráster cuando están disponibles, excluye NoData y calcula la normalización y los pesos de clase únicamente con el conjunto de entrenamiento.

In [ ]:
def read_reflectance(path: str | Path, band_indexes=None):
    band_indexes = BAND_INDEXES if band_indexes is None else band_indexes

    with rasterio.open(path) as src:
        array = src.read(indexes=band_indexes).astype(np.float32)
        nodata = src.nodata
        scales = np.asarray(
            [src.scales[index - 1] for index in band_indexes],
            dtype=np.float32,
        )[:, None, None]
        offsets = np.asarray(
            [src.offsets[index - 1] for index in band_indexes],
            dtype=np.float32,
        )[:, None, None]

    finite = np.isfinite(array)
    if nodata is not None and np.isfinite(nodata):
        finite &= array != nodata

    valid_pixel = finite.all(axis=0)
    array = np.where(finite, array, np.nan)

    finite_values = array[np.isfinite(array)]
    if finite_values.size == 0:
        raise ValueError(f"La imagen no contiene valores válidos: {path}")

    has_raster_transform = (
        USE_RASTER_SCALE_OFFSET
        and (
            not np.allclose(scales, 1.0)
            or not np.allclose(offsets, 0.0)
        )
    )
    if has_raster_transform:
        array = array * scales + offsets
    elif AUTO_SCALE_REFLECTANCE and float(np.nanpercentile(finite_values, 99.5)) > 2.0:
        array = (
            array + float(REFLECTANCE_ADD_OFFSET)
        ) / float(REFLECTANCE_SCALE)

    return np.clip(array, 0.0, 1.0).astype(np.float32), valid_pixel


class LULCSegmentationDataset(Dataset):
    def __init__(
        self,
        dataframe: pd.DataFrame,
        code_to_idx: dict[int, int],
        means: list[float],
        stds: list[float],
        augment: bool = False,
    ):
        self.df = dataframe.reset_index(drop=True)
        self.code_to_idx = code_to_idx
        self.means = np.asarray(means, dtype=np.float32)[:, None, None]
        self.stds = np.asarray(stds, dtype=np.float32)[:, None, None]
        self.augment = augment

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, index: int):
        row = self.df.iloc[index]
        image, valid_pixel = read_reflectance(row["image_path"])

        with rasterio.open(row["mask_path"]) as src:
            original_mask = src.read(1).astype(np.int64)

        if image.shape[1:] != original_mask.shape:
            raise ValueError(
                f"Dimensiones incompatibles: {row['image_path']} y {row['mask_path']}."
            )

        mask = np.full(original_mask.shape, IGNORE_INDEX, dtype=np.int64)
        for original_code, internal_index in self.code_to_idx.items():
            mask[(original_mask == original_code) & valid_pixel] = internal_index

        if self.augment:
            if random.random() < 0.5:
                image = np.flip(image, axis=2)
                mask = np.flip(mask, axis=1)
            if random.random() < 0.5:
                image = np.flip(image, axis=1)
                mask = np.flip(mask, axis=0)
            rotation = random.randint(0, 3)
            if rotation:
                image = np.rot90(image, rotation, axes=(1, 2))
                mask = np.rot90(mask, rotation, axes=(0, 1))

        valid = np.isfinite(image).all(axis=0) & (mask != IGNORE_INDEX)
        image = np.nan_to_num(image, nan=0.0)
        image = (image - self.means) / np.maximum(self.stds, 1e-6)
        image[:, ~valid] = 0.0

        return {
            "image": torch.from_numpy(np.ascontiguousarray(image)).float(),
            "mask": torch.from_numpy(np.ascontiguousarray(mask)).long(),
            "image_path": row["image_path"],
            "mask_path": row["mask_path"],
        }


def compute_image_stats(
    dataframe: pd.DataFrame,
    sample_size: int | None = None,
) -> tuple[list[float], list[float]]:
    sample_size = STATS_SAMPLE_SIZE if sample_size is None else sample_size
    paths = dataframe["image_path"].tolist()

    if sample_size and len(paths) > sample_size:
        rng = np.random.default_rng(SEED)
        paths = rng.choice(paths, size=sample_size, replace=False).tolist()

    channel_sum = None
    channel_squared_sum = None
    channel_count = None

    for path in tqdm(paths, desc="Estadísticas radiométricas"):
        array, _ = read_reflectance(path)
        flat = array.reshape(array.shape[0], -1)
        valid = np.isfinite(flat)

        sums = np.nansum(flat, axis=1)
        squared_sums = np.nansum(flat * flat, axis=1)
        counts = valid.sum(axis=1)

        if channel_sum is None:
            channel_sum = sums
            channel_squared_sum = squared_sums
            channel_count = counts
        else:
            channel_sum += sums
            channel_squared_sum += squared_sums
            channel_count += counts

    if channel_sum is None or np.any(channel_count == 0):
        raise RuntimeError("No fue posible calcular las estadísticas radiométricas.")

    means = channel_sum / channel_count
    variances = channel_squared_sum / channel_count - means**2
    stds = np.sqrt(np.maximum(variances, 1e-12))
    return means.astype(float).tolist(), stds.astype(float).tolist()


def compute_class_pixel_counts(
    dataframe: pd.DataFrame,
    code_to_idx: dict[int, int],
) -> np.ndarray:
    counts = np.zeros(len(code_to_idx), dtype=np.int64)

    iterator = dataframe[["image_path", "mask_path"]].itertuples(index=False)
    for image_path, mask_path in tqdm(
        iterator,
        total=len(dataframe),
        desc="Píxeles por clase",
    ):
        _, valid_pixel = read_reflectance(image_path)
        with rasterio.open(mask_path) as src:
            mask = src.read(1)

        if mask.shape != valid_pixel.shape:
            raise ValueError(
                f"Dimensiones incompatibles al contar clases: {image_path} y {mask_path}."
            )

        for original_code, internal_index in code_to_idx.items():
            counts[internal_index] += int(
                ((mask == original_code) & valid_pixel).sum()
            )

    return counts


def make_class_weights(counts: np.ndarray) -> torch.Tensor:
    if np.any(counts == 0):
        missing = np.where(counts == 0)[0].tolist()
        raise ValueError(f"Clases internas sin píxeles de entrenamiento: {missing}")

    frequencies = counts / counts.sum()
    weights = 1.0 / np.sqrt(frequencies)
    weights /= weights.mean()
    return torch.tensor(weights, dtype=torch.float32)


def build_weighted_sampler(dataframe: pd.DataFrame):
    if not USE_WEIGHTED_SAMPLER:
        return None
    if "class_code" not in dataframe.columns:
        warnings.warn(
            "No existe class_code; se omite el muestreo balanceado por chip.",
            RuntimeWarning,
        )
        return None

    codes = pd.to_numeric(dataframe["class_code"], errors="coerce").fillna(-1).astype(int)
    counts = codes.value_counts().to_dict()
    sample_weights = codes.map(lambda code: 1.0 / counts[int(code)]).to_numpy(dtype=np.float64)

    generator = torch.Generator()
    generator.manual_seed(SEED)
    return WeightedRandomSampler(
        torch.as_tensor(sample_weights, dtype=torch.double),
        num_samples=len(sample_weights),
        replacement=True,
        generator=generator,
    )


def save_dataset_summary(
    train: pd.DataFrame,
    val: pd.DataFrame,
    test: pd.DataFrame,
    pixel_counts: np.ndarray,
    idx_to_code: dict[int, int],
    class_dict: dict[int, str],
    output_dir: Path,
) -> None:
    output_dir = ensure_dir(output_dir)

    split_summary = pd.DataFrame(
        {
            "split": ["train", "val", "test"],
            "chips": [len(train), len(val), len(test)],
        }
    )
    split_summary["percentage"] = (
        split_summary["chips"] / split_summary["chips"].sum() * 100.0
    )
    split_summary.to_csv(
        output_dir / "dataset_splits.csv",
        index=False,
        encoding="utf-8-sig",
    )

    pixel_summary = pd.DataFrame(
        {
            "class_index": np.arange(len(pixel_counts)),
            "class_code": [idx_to_code[index] for index in range(len(pixel_counts))],
            "class_name": [
                class_dict[idx_to_code[index]]
                for index in range(len(pixel_counts))
            ],
            "pixels_train": pixel_counts,
        }
    )
    pixel_summary["percentage_train"] = (
        pixel_summary["pixels_train"] / max(pixel_summary["pixels_train"].sum(), 1) * 100.0
    )
    pixel_summary.to_csv(
        output_dir / "dataset_pixels_por_clase.csv",
        index=False,
        encoding="utf-8-sig",
    )

## Arquitectura

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, dropout: float = 0.0):
        super().__init__()
        layers = [
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
        ]
        if dropout > 0:
            layers.insert(2, nn.Dropout2d(dropout))
        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)


class Down(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, dropout: float = 0.0):
        super().__init__()
        self.block = nn.Sequential(
            nn.MaxPool2d(kernel_size=2),
            DoubleConv(in_channels, out_channels, dropout),
        )

    def forward(self, x):
        return self.block(x)


class Up(nn.Module):
    def __init__(
        self,
        in_channels: int,
        skip_channels: int,
        out_channels: int,
        bilinear: bool = False,
        dropout: float = 0.0,
    ):
        super().__init__()
        if bilinear:
            self.up = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False)
            convolution_channels = in_channels + skip_channels
        else:
            self.up = nn.ConvTranspose2d(
                in_channels,
                out_channels,
                kernel_size=2,
                stride=2,
            )
            convolution_channels = out_channels + skip_channels

        self.conv = DoubleConv(convolution_channels, out_channels, dropout)

    def forward(self, decoder, encoder):
        decoder = self.up(decoder)
        if decoder.shape[-2:] != encoder.shape[-2:]:
            decoder = F.interpolate(
                decoder,
                size=encoder.shape[-2:],
                mode="bilinear",
                align_corners=False,
            )
        return self.conv(torch.cat([encoder, decoder], dim=1))


class UNet(nn.Module):
    def __init__(
        self,
        in_channels: int,
        num_classes: int,
        base_channels: int = BASE_CHANNELS,
        bilinear: bool = BILINEAR,
        dropout: float = DROPOUT,
        bottleneck_dropout: float = BOTTLENECK_DROPOUT,
    ):
        super().__init__()
        self.inc = DoubleConv(in_channels, base_channels, dropout)
        self.down1 = Down(base_channels, base_channels * 2, dropout)
        self.down2 = Down(base_channels * 2, base_channels * 4, dropout)
        self.down3 = Down(base_channels * 4, base_channels * 8, dropout)
        self.down4 = Down(base_channels * 8, base_channels * 16, bottleneck_dropout)

        self.up1 = Up(base_channels * 16, base_channels * 8, base_channels * 8, bilinear, dropout)
        self.up2 = Up(base_channels * 8, base_channels * 4, base_channels * 4, bilinear, dropout)
        self.up3 = Up(base_channels * 4, base_channels * 2, base_channels * 2, bilinear, dropout)
        self.up4 = Up(base_channels * 2, base_channels, base_channels, bilinear, dropout)
        self.out = nn.Conv2d(base_channels, num_classes, kernel_size=1)

        self.apply(self._initialize_weights)

    @staticmethod
    def _initialize_weights(module):
        if isinstance(module, (nn.Conv2d, nn.ConvTranspose2d)):
            nn.init.kaiming_normal_(module.weight, mode="fan_in", nonlinearity="relu")
            if module.bias is not None:
                nn.init.zeros_(module.bias)

    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)

        x = self.up1(x5, x4)
        x = self.up2(x, x3)
        x = self.up3(x, x2)
        x = self.up4(x, x1)
        return self.out(x)

## Función de pérdida y métricas

In [ ]:
class DiceLoss(nn.Module):
    def __init__(self, num_classes: int, ignore_index: int = IGNORE_INDEX, smooth: float = 1.0):
        super().__init__()
        self.num_classes = num_classes
        self.ignore_index = ignore_index
        self.smooth = smooth

    def forward(self, logits, target):
        probabilities = torch.softmax(logits, dim=1)
        valid = target != self.ignore_index
        losses = []

        for class_index in range(self.num_classes):
            if class_index == self.ignore_index:
                continue

            truth = ((target == class_index) & valid).float()
            if truth.sum() == 0:
                continue

            prediction = probabilities[:, class_index] * valid
            intersection = (prediction * truth).sum()
            denominator = prediction.sum() + truth.sum()
            dice = (2.0 * intersection + self.smooth) / (denominator + self.smooth)
            losses.append(1.0 - dice)

        return torch.stack(losses).mean() if losses else logits.sum() * 0.0


class CombinedSegmentationLoss(nn.Module):
    def __init__(
        self,
        num_classes: int,
        class_weights: torch.Tensor | None,
    ):
        super().__init__()
        self.cross_entropy = nn.CrossEntropyLoss(
            weight=class_weights,
            ignore_index=IGNORE_INDEX,
        )
        self.dice = DiceLoss(num_classes, IGNORE_INDEX)

    def forward(self, logits, target):
        ce_loss = self.cross_entropy(logits, target)
        if LOSS_TYPE == "weighted_ce":
            return ce_loss
        if LOSS_TYPE == "ce_dice":
            return ce_loss + DICE_WEIGHT * self.dice(logits, target)
        raise ValueError(f"LOSS_TYPE no reconocido: {LOSS_TYPE}")


class ConfusionAccumulator:
    def __init__(self, num_classes: int, ignore_index: int = IGNORE_INDEX):
        self.num_classes = num_classes
        self.ignore_index = ignore_index
        self.matrix = torch.zeros((num_classes, num_classes), dtype=torch.int64)

    @torch.no_grad()
    def update(self, logits, target) -> None:
        prediction = logits.argmax(dim=1)
        valid = (
            (target != self.ignore_index)
            & (target >= 0)
            & (target < self.num_classes)
            & (prediction >= 0)
            & (prediction < self.num_classes)
        )
        encoded = self.num_classes * target[valid] + prediction[valid]
        counts = torch.bincount(
            encoded.detach().cpu(),
            minlength=self.num_classes**2,
        )
        self.matrix += counts.reshape(self.num_classes, self.num_classes)


def metrics_from_confusion(matrix: np.ndarray, ignore_index: int = IGNORE_INDEX):
    matrix = matrix.astype(np.float64)
    true_positive = np.diag(matrix)
    support = matrix.sum(axis=1)
    predicted = matrix.sum(axis=0)
    total = matrix.sum()

    precision = np.divide(
        true_positive,
        predicted,
        out=np.zeros_like(true_positive),
        where=predicted > 0,
    )
    recall = np.divide(
        true_positive,
        support,
        out=np.zeros_like(true_positive),
        where=support > 0,
    )
    f1 = np.divide(
        2 * precision * recall,
        precision + recall,
        out=np.zeros_like(true_positive),
        where=(precision + recall) > 0,
    )
    iou = np.divide(
        true_positive,
        support + predicted - true_positive,
        out=np.zeros_like(true_positive),
        where=(support + predicted - true_positive) > 0,
    )

    active = (support > 0) & (np.arange(len(support)) != ignore_index)
    pixel_accuracy = float(true_positive.sum() / total) if total else 0.0
    expected = float((support * predicted).sum() / (total**2)) if total else 0.0
    kappa = (
        (pixel_accuracy - expected) / (1.0 - expected)
        if total and expected < 1.0
        else 0.0
    )

    summary = {
        "pixel_accuracy": pixel_accuracy,
        "macro_precision": float(precision[active].mean()) if active.any() else 0.0,
        "macro_recall": float(recall[active].mean()) if active.any() else 0.0,
        "macro_f1": float(f1[active].mean()) if active.any() else 0.0,
        "mean_iou": float(iou[active].mean()) if active.any() else 0.0,
        "kappa": float(kappa),
    }
    per_class = pd.DataFrame(
        {
            "class_code": np.arange(len(support)),
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "iou": iou,
            "support": support.astype(np.int64),
        }
    )
    return summary, per_class


def amp_enabled() -> bool:
    return bool(USE_AMP and DEVICE.type == "cuda")


def autocast_context():
    if amp_enabled():
        return torch.autocast(device_type="cuda", dtype=torch.float16)
    return nullcontext()


def make_grad_scaler():
    try:
        return torch.amp.GradScaler("cuda", enabled=amp_enabled())
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=amp_enabled())


def train_one_epoch(model, loader, optimizer, criterion, scaler, epoch, num_classes):
    model.train()
    accumulator = ConfusionAccumulator(num_classes)
    total_loss = 0.0
    total_valid_pixels = 0
    optimizer.zero_grad(set_to_none=True)

    progress = tqdm(loader, desc=f"Entrenamiento {epoch}")
    for step, batch in enumerate(progress, start=1):
        images = batch["image"].to(DEVICE, non_blocking=True)
        masks = batch["mask"].to(DEVICE, non_blocking=True)

        group_start = (
            (step - 1) // GRADIENT_ACCUMULATION_STEPS
        ) * GRADIENT_ACCUMULATION_STEPS
        group_size = min(
            GRADIENT_ACCUMULATION_STEPS,
            len(loader) - group_start,
        )

        with autocast_context():
            logits = model(images)
            loss = criterion(logits, masks)
            scaled_loss = loss / group_size

        if amp_enabled():
            scaler.scale(scaled_loss).backward()
        else:
            scaled_loss.backward()

        should_step = (
            step % GRADIENT_ACCUMULATION_STEPS == 0
            or step == len(loader)
        )
        if should_step:
            if amp_enabled():
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()
            optimizer.zero_grad(set_to_none=True)

        valid_pixels = int((masks != IGNORE_INDEX).sum().item())
        total_loss += float(loss.item()) * max(valid_pixels, 1)
        total_valid_pixels += valid_pixels
        accumulator.update(logits.detach(), masks)

        metrics, _ = metrics_from_confusion(accumulator.matrix.numpy())
        progress.set_postfix(
            loss=f"{total_loss / max(total_valid_pixels, 1):.4f}",
            mIoU=f"{metrics['mean_iou']:.4f}",
        )

    metrics, _ = metrics_from_confusion(accumulator.matrix.numpy())
    metrics["loss"] = total_loss / max(total_valid_pixels, 1)
    return metrics


@torch.no_grad()
def evaluate(model, loader, criterion, split_name, num_classes):
    model.eval()
    accumulator = ConfusionAccumulator(num_classes)
    total_loss = 0.0
    total_valid_pixels = 0

    progress = tqdm(loader, desc=split_name)
    for batch in progress:
        images = batch["image"].to(DEVICE, non_blocking=True)
        masks = batch["mask"].to(DEVICE, non_blocking=True)

        with autocast_context():
            logits = model(images)
            loss = criterion(logits, masks)

        valid_pixels = int((masks != IGNORE_INDEX).sum().item())
        total_loss += float(loss.item()) * max(valid_pixels, 1)
        total_valid_pixels += valid_pixels
        accumulator.update(logits, masks)

        metrics, _ = metrics_from_confusion(accumulator.matrix.numpy())
        progress.set_postfix(
            loss=f"{total_loss / max(total_valid_pixels, 1):.4f}",
            mIoU=f"{metrics['mean_iou']:.4f}",
        )

    metrics, per_class = metrics_from_confusion(accumulator.matrix.numpy())
    metrics["loss"] = total_loss / max(total_valid_pixels, 1)
    return metrics, per_class, accumulator.matrix.numpy()

## Checkpoints y reportes

In [ ]:
def atomic_torch_save(payload, path: str | Path) -> None:
    path = Path(path)
    ensure_dir(path.parent)
    temporary = path.with_suffix(path.suffix + ".tmp")
    torch.save(payload, temporary)
    os.replace(temporary, path)


def save_checkpoint(
    path,
    epoch,
    model,
    optimizer,
    scheduler,
    scaler,
    config,
    validation_metrics,
    best_value,
    best_epoch,
    patience,
):
    atomic_torch_save(
        {
            "epoch": int(epoch),
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "scaler_state_dict": scaler.state_dict() if scaler is not None else None,
            "config": config,
            "validation_metrics": validation_metrics,
            "best_val_iou": float(best_value),
            "best_epoch": int(best_epoch),
            "patience": int(patience),
            "saved_at": datetime.now().isoformat(timespec="seconds"),
        },
        path,
    )


def load_checkpoint_if_needed(
    last_checkpoint,
    model,
    optimizer,
    scheduler,
    scaler,
    auto_resume,
):
    if not auto_resume or not Path(last_checkpoint).exists():
        return 1, float("-inf"), 0, 0

    try:
        checkpoint = torch.load(last_checkpoint, map_location=DEVICE, weights_only=False)
    except TypeError:
        checkpoint = torch.load(last_checkpoint, map_location=DEVICE)

    model.load_state_dict(checkpoint["model_state_dict"], strict=True)

    for key, target in (
        ("optimizer_state_dict", optimizer),
        ("scheduler_state_dict", scheduler),
    ):
        state = checkpoint.get(key)
        if state:
            try:
                target.load_state_dict(state)
            except Exception as exc:
                warnings.warn(f"No se restauró {key}: {exc}", RuntimeWarning)

    scaler_state = checkpoint.get("scaler_state_dict")
    if scaler is not None and isinstance(scaler_state, dict) and scaler_state:
        try:
            scaler.load_state_dict(scaler_state)
        except Exception as exc:
            warnings.warn(f"No se restauró GradScaler: {exc}", RuntimeWarning)

    return (
        int(checkpoint.get("epoch", 0)) + 1,
        float(checkpoint.get("best_val_iou", float("-inf"))),
        int(checkpoint.get("best_epoch", 0)),
        int(checkpoint.get("patience", 0)),
    )


def plot_history(history: pd.DataFrame, output_dir: Path) -> None:
    if history.empty:
        return

    output_dir = ensure_dir(output_dir)
    pairs = [
        ("train_loss", "val_loss", "Pérdida"),
        ("train_pixel_accuracy", "val_pixel_accuracy", "Exactitud por píxel"),
        ("train_mean_iou", "val_mean_iou", "IoU media"),
        ("train_macro_f1", "val_macro_f1", "F1 macro"),
        ("train_kappa", "val_kappa", "Kappa"),
    ]

    for train_column, val_column, title in pairs:
        plt.figure(figsize=(9, 5))
        if train_column in history:
            plt.plot(history["epoch"], history[train_column], label="Entrenamiento")
        if val_column in history:
            plt.plot(history["epoch"], history[val_column], label="Validación")
        plt.xlabel("Época")
        plt.ylabel(title)
        plt.title(title)
        plt.grid(True)
        plt.legend()
        plt.tight_layout()
        plt.savefig(output_dir / f"{safe_name(title)}.png", dpi=180)
        plt.close()


def plot_confusion_matrix(matrix, idx_to_name, path):
    labels = list(range(matrix.shape[0]))
    names = [idx_to_name[index] for index in labels]

    plt.figure(figsize=(10, 8))
    plt.imshow(matrix, interpolation="nearest")
    plt.colorbar()
    ticks = np.arange(len(labels))
    plt.xticks(ticks, names, rotation=90)
    plt.yticks(ticks, names)
    plt.xlabel("Predicción")
    plt.ylabel("Referencia")
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.close()


def save_test_predictions(
    model,
    loader,
    output_dir,
    max_items,
    idx_to_code: dict[int, int],
):
    output_dir = ensure_dir(output_dir)
    model.eval()
    saved = 0
    lookup = np.asarray(
        [idx_to_code[index] for index in range(len(idx_to_code))],
        dtype=np.uint8,
    )

    with torch.no_grad():
        for batch in tqdm(loader, desc="Predicciones de prueba"):
            images = batch["image"].to(DEVICE)
            with autocast_context():
                internal_predictions = model(images).argmax(dim=1).cpu().numpy()

            for index, internal_prediction in enumerate(internal_predictions):
                if saved >= max_items:
                    return

                image_path = batch["image_path"][index]
                prediction = lookup[internal_prediction]
                _, valid_pixel = read_reflectance(image_path)
                prediction = prediction.copy()
                prediction[~valid_pixel] = MASK_NODATA_CODE

                with rasterio.open(image_path) as src:
                    profile = src.profile.copy()
                profile.update(
                    count=1,
                    dtype="uint8",
                    nodata=MASK_NODATA_CODE,
                    compress="lzw",
                )

                output = output_dir / f"prediccion_test_{saved:04d}.tif"
                with rasterio.open(output, "w", **profile) as dst:
                    dst.write(prediction, 1)
                saved += 1


def write_markdown_report(summary, config, path):
    lines = ["# Informe de entrenamiento U-Net\n", "## Configuración\n"]
    for key, value in config.items():
        lines.append(f"- **{key}**: {value}")

    lines.append("\n## Resultados de prueba\n")
    for key, value in summary.items():
        formatted = f"{float(value):.6f}" if isinstance(value, (int, float, np.integer, np.floating)) else str(value)
        lines.append(f"- **{key}**: {formatted}")

    path = Path(path)
    ensure_dir(path.parent)
    path.write_text("\n".join(lines), encoding="utf-8")

## Entrenamiento por nivel

In [ ]:
def build_run_dir(level_name: str):
    forced = RESUME_BY_LEVEL.get(level_name)

    if forced:
        experiment_name = forced
        auto_resume = True
    elif CREATE_NEW_RUNS:
        experiment_name = f"UNet_{level_name}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
        auto_resume = False
    else:
        candidates = []
        if RUNS_DIR.exists():
            for run_dir in RUNS_DIR.glob(f"UNet_{level_name}_*"):
                checkpoint = run_dir / "checkpoints" / "last_checkpoint.pt"
                if checkpoint.exists():
                    candidates.append((checkpoint.stat().st_mtime, run_dir.name))

        if candidates:
            experiment_name = max(candidates)[1]
            auto_resume = True
        else:
            experiment_name = f"UNet_{level_name}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
            auto_resume = False

    run_dir = ensure_dir(RUNS_DIR / experiment_name)
    checkpoint_dir = ensure_dir(run_dir / "checkpoints")
    report_dir = ensure_dir(run_dir / "reports")
    prediction_dir = ensure_dir(run_dir / "test_predictions")
    return run_dir, checkpoint_dir, report_dir, prediction_dir, experiment_name, auto_resume


def train_level(level_name: str):
    class_dict = get_class_dict(level_name)
    thematic_codes = sorted(code for code in class_dict if code != MASK_NODATA_CODE)
    code_to_idx = {code: index for index, code in enumerate(thematic_codes)}
    idx_to_code = {index: code for code, index in code_to_idx.items()}
    idx_to_name = {
        index: class_dict[code]
        for index, code in idx_to_code.items()
    }
    num_classes = len(code_to_idx)

    train_df, val_df, test_df, splits_dir, band_count, chip_shape = read_splits(
        level_name,
        class_dict,
    )

    means, stds = compute_image_stats(train_df)
    pixel_counts = compute_class_pixel_counts(train_df, code_to_idx)
    class_weights = (
        make_class_weights(pixel_counts).to(DEVICE)
        if USE_CLASS_WEIGHTS
        else None
    )
    sampler = build_weighted_sampler(train_df)

    if USE_CLASS_WEIGHTS and USE_WEIGHTED_SAMPLER:
        warnings.warn(
            "Se están aplicando simultáneamente pesos de clase y muestreo balanceado.",
            RuntimeWarning,
        )

    train_loader = DataLoader(
        LULCSegmentationDataset(
            train_df,
            code_to_idx,
            means,
            stds,
            augment=USE_AUGMENTATION,
        ),
        batch_size=BATCH_SIZE,
        shuffle=sampler is None,
        sampler=sampler,
        num_workers=NUM_WORKERS,
        pin_memory=DEVICE.type == "cuda",
        persistent_workers=NUM_WORKERS > 0,
    )
    val_loader = DataLoader(
        LULCSegmentationDataset(val_df, code_to_idx, means, stds, augment=False),
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=DEVICE.type == "cuda",
        persistent_workers=NUM_WORKERS > 0,
    )
    test_loader = DataLoader(
        LULCSegmentationDataset(test_df, code_to_idx, means, stds, augment=False),
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=DEVICE.type == "cuda",
        persistent_workers=NUM_WORKERS > 0,
    )

    model = UNet(
        in_channels=len(BAND_INDEXES),
        num_classes=num_classes,
        base_channels=BASE_CHANNELS,
        bilinear=BILINEAR,
        dropout=DROPOUT,
        bottleneck_dropout=BOTTLENECK_DROPOUT,
    ).to(DEVICE)

    first_batch = next(iter(train_loader))["image"].to(DEVICE)
    with torch.no_grad():
        output = model(first_batch[:1])
    expected_output = (1, num_classes, first_batch.shape[-2], first_batch.shape[-1])
    if tuple(output.shape) != expected_output:
        raise RuntimeError(
            f"Salida inesperada del modelo: {tuple(output.shape)}; "
            f"se esperaba {expected_output}."
        )

    if OPTIMIZER_TYPE == "adamw":
        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=LEARNING_RATE,
            weight_decay=WEIGHT_DECAY,
        )
    elif OPTIMIZER_TYPE == "sgd":
        optimizer = torch.optim.SGD(
            model.parameters(),
            lr=LEARNING_RATE,
            momentum=0.99,
            weight_decay=WEIGHT_DECAY,
        )
    else:
        raise ValueError(f"OPTIMIZER_TYPE no reconocido: {OPTIMIZER_TYPE}")

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=8,
        min_lr=1e-6,
    )
    criterion = CombinedSegmentationLoss(num_classes, class_weights)
    scaler = make_grad_scaler()

    (
        run_dir,
        checkpoint_dir,
        report_dir,
        prediction_dir,
        experiment_name,
        auto_resume,
    ) = build_run_dir(level_name)

    config = {
        "model": "U-Net",
        "task": "segmentacion_semantica",
        "level": level_name,
        "dataset_root": str(DATASET_ROOT),
        "splits_dir": str(splits_dir),
        "run_dir": str(run_dir),
        "experiment_name": experiment_name,
        "band_indexes": list(BAND_INDEXES),
        "band_order": ["B2", "B3", "B4", "B8"],
        "reflectance_scale": REFLECTANCE_SCALE,
        "reflectance_add_offset": REFLECTANCE_ADD_OFFSET,
        "use_raster_scale_offset": USE_RASTER_SCALE_OFFSET,
        "auto_scale_reflectance": AUTO_SCALE_REFLECTANCE,
        "band_count": band_count,
        "chip_shape": chip_shape,
        "num_classes": num_classes,
        "mask_nodata_code": MASK_NODATA_CODE,
        "ignore_index": IGNORE_INDEX,
        "code_to_idx": code_to_idx,
        "idx_to_code": idx_to_code,
        "idx_to_name": idx_to_name,
        "means": means,
        "stds": stds,
        "base_channels": BASE_CHANNELS,
        "bilinear": BILINEAR,
        "dropout": DROPOUT,
        "bottleneck_dropout": BOTTLENECK_DROPOUT,
        "optimizer": OPTIMIZER_TYPE,
        "loss": LOSS_TYPE,
        "dice_weight": DICE_WEIGHT,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "use_class_weights": USE_CLASS_WEIGHTS,
        "use_weighted_sampler": USE_WEIGHTED_SAMPLER,
        "use_augmentation": USE_AUGMENTATION,
        "seed": SEED,
    }
    save_json(config, run_dir / "config.json")
    save_dataset_summary(
        train_df,
        val_df,
        test_df,
        pixel_counts,
        idx_to_code,
        class_dict,
        report_dir / "dataset",
    )

    history_path = run_dir / "history.csv"
    history = (
        pd.read_csv(history_path).to_dict("records")
        if history_path.exists() and history_path.stat().st_size > 0
        else []
    )

    last_checkpoint = checkpoint_dir / "last_checkpoint.pt"
    start_epoch, best_value, best_epoch, patience = load_checkpoint_if_needed(
        last_checkpoint,
        model,
        optimizer,
        scheduler,
        scaler,
        auto_resume,
    )

    for epoch in range(start_epoch, EPOCHS + 1):
        train_metrics = train_one_epoch(
            model,
            train_loader,
            optimizer,
            criterion,
            scaler,
            epoch,
            num_classes,
        )
        val_metrics, _, _ = evaluate(
            model,
            val_loader,
            criterion,
            f"Validación {level_name} época {epoch}",
            num_classes,
        )

        scheduler.step(val_metrics["mean_iou"])
        improved = val_metrics["mean_iou"] > best_value

        if improved:
            best_value = val_metrics["mean_iou"]
            best_epoch = epoch
            patience = 0
        else:
            patience += 1

        row = {"epoch": epoch, "learning_rate": optimizer.param_groups[0]["lr"]}
        row.update({f"train_{key}": value for key, value in train_metrics.items()})
        row.update({f"val_{key}": value for key, value in val_metrics.items()})
        history.append(row)

        history_df = (
            pd.DataFrame(history)
            .drop_duplicates(subset=["epoch"], keep="last")
            .sort_values("epoch")
        )
        history_df.to_csv(history_path, index=False, encoding="utf-8-sig")

        save_checkpoint(
            last_checkpoint,
            epoch,
            model,
            optimizer,
            scheduler,
            scaler,
            config,
            val_metrics,
            best_value,
            best_epoch,
            patience,
        )

        if improved:
            save_checkpoint(
                run_dir / "best_model.pt",
                epoch,
                model,
                optimizer,
                scheduler,
                scaler,
                config,
                val_metrics,
                best_value,
                best_epoch,
                patience,
            )

        if SAVE_EVERY_EPOCH:
            save_checkpoint(
                checkpoint_dir / f"checkpoint_epoch_{epoch:03d}.pt",
                epoch,
                model,
                optimizer,
                scheduler,
                scaler,
                config,
                val_metrics,
                best_value,
                best_epoch,
                patience,
            )

        if SAVE_REPORT_PLOTS:
            plot_history(history_df, report_dir)

        print(
            f"[{level_name} | época {epoch}] "
            f"train_loss={train_metrics['loss']:.4f} "
            f"val_loss={val_metrics['loss']:.4f} "
            f"val_mIoU={val_metrics['mean_iou']:.4f} "
            f"mejor_mIoU={best_value:.4f} "
            f"paciencia={patience}/{EARLY_STOPPING_PATIENCE}"
        )

        if patience >= EARLY_STOPPING_PATIENCE:
            break

    best_path = run_dir / "best_model.pt"
    checkpoint_path = best_path if best_path.exists() else last_checkpoint
    if not checkpoint_path.exists():
        raise FileNotFoundError(f"No se generó un checkpoint en {run_dir}")

    try:
        checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)
    except TypeError:
        checkpoint = torch.load(checkpoint_path, map_location=DEVICE)
    model.load_state_dict(checkpoint["model_state_dict"], strict=True)

    test_metrics, per_class, confusion = evaluate(
        model,
        test_loader,
        criterion,
        f"Prueba {level_name}",
        num_classes,
    )

    per_class = per_class.rename(columns={"class_code": "class_index"})
    per_class["class_code"] = per_class["class_index"].map(idx_to_code)
    per_class["class_name"] = per_class["class_index"].map(idx_to_name)
    per_class.to_csv(
        report_dir / "metricas_por_clase_test.csv",
        index=False,
        encoding="utf-8-sig",
    )

    labels = list(range(num_classes))
    names = [idx_to_name[index] for index in labels]
    pd.DataFrame(
        confusion,
        index=names,
        columns=names,
    ).to_csv(
        report_dir / "matriz_confusion_test.csv",
        encoding="utf-8-sig",
    )

    if SAVE_REPORT_PLOTS:
        plot_confusion_matrix(
            confusion,
            idx_to_name,
            report_dir / "matriz_confusion_test.png",
        )

    if SAVE_TEST_PREDICTIONS:
        save_test_predictions(
            model,
            test_loader,
            prediction_dir,
            MAX_TEST_PREDICTIONS_TO_SAVE,
            idx_to_code,
        )

    summary = {
        **test_metrics,
        "train_level": level_name,
        "run_dir": str(run_dir),
        "best_epoch": int(best_epoch),
        "best_val_iou": float(best_value),
    }
    save_json(summary, report_dir / "resumen_metricas_test.json")
    write_markdown_report(
        summary,
        config,
        report_dir / "informe_entrenamiento_unet.md",
    )

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return summary


def run_all_levels():
    set_seed(SEED)
    ensure_dir(RUNS_DIR)

    summaries = [train_level(level_name) for level_name in LEVELS_TO_TRAIN]
    summary_df = pd.DataFrame(summaries)
    output = RUNS_DIR / (
        f"resumen_unet_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
    )
    summary_df.to_csv(output, index=False, encoding="utf-8-sig")
    print(summary_df)
    print(f"Resumen general: {output}")
    return summary_df

## Ejecución

Revise las rutas y los parámetros antes de ejecutar esta celda. Las muestras de nubes se entrenan de forma independiente y no se incorporan como una categoría CLC sin un remapeo explícito.

In [ ]:
if REQUIRE_CUDA and DEVICE.type != "cuda":
    raise RuntimeError(
        "CUDA no está disponible; cambia REQUIRE_CUDA a False para ejecutar en CPU."
    )

summary_df = run_all_levels()